3 - Entrenar y validar un modelo para predecir la duración de la estadía. Para esta tarea, necesitarás elegir un tamaño de ventana para entrenar tus datos (demasiados días retrasarán las decisiones sobre pacientes cuando el sistema se implemente, muy pocos días probablemente producirán un predictor muy miope. Elige con cuidado).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV
from sklearn import preprocessing
from sklearn.metrics import mean_absolute_error


# Cargar tablas MIMIC-III
base_path = 'C:/Users/Ines/Desktop/Nova_entrega/Parte_topicos/'

icustays_cleaned = pd.read_excel(f'{base_path}icustays_cleaned.xlsx')
diagnoses_icd_cleaned = pd.read_excel(f'{base_path}diagnoses_icd_cleaned.xlsx')
chartevents = pd.read_csv(f'{base_path}CHARTEVENTS.csv')

C:\Users\Ines\AppData\Local\Temp\ipykernel_868\3242706356.py:30: DtypeWarning: Columns (8,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  chartevents = pd.read_csv(f'{base_path}CHARTEVENTS.csv')


In [2]:
chartevents.info(verbose=True, show_counts=True)
chartevents.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39195630 entries, 0 to 39195629
Data columns (total 15 columns):
 #   Column        Non-Null Count     Dtype  
---  ------        --------------     -----  
 0   ROW_ID        39195630 non-null  int64  
 1   SUBJECT_ID    39195630 non-null  int64  
 2   HADM_ID       39195630 non-null  int64  
 3   ICUSTAY_ID    39161239 non-null  float64
 4   ITEMID        39195630 non-null  int64  
 5   CHARTTIME     39195630 non-null  object 
 6   STORETIME     32882957 non-null  object 
 7   CGID          32882957 non-null  float64
 8   VALUE         38919020 non-null  object 
 9   VALUENUM      36320500 non-null  float64
 10  VALUEUOM      34093161 non-null  object 
 11  WARNING       34303752 non-null  float64
 12  ERROR         34303752 non-null  float64
 13  RESULTSTATUS  289761 non-null    object 
 14  STOPPED       4864645 non-null   object 
dtypes: float64(5), int64(4), object(6)
memory usage: 4.4+ GB


,ROW_ID,SUBJECT_ID,HADM_ID,ICUSTAY_ID,ITEMID,CHARTTIME,STORETIME,CGID,VALUE,VALUENUM,VALUEUOM,WARNING,ERROR,RESULTSTATUS,STOPPED
0,788,36,165660,241249.0,223834,2134-05-12 12:00:00,2134-05-12 13:56:00,17525.0,15.0,15.00,L/min,0.0,0.0,NaN,NaN
1,789,36,165660,241249.0,223835,2134-05-12 12:00:00,2134-05-12 13:56:00,17525.0,100.0,100.00,NaN,0.0,0.0,NaN,NaN
2,790,36,165660,241249.0,224328,2134-05-12 12:00:00,2134-05-12 12:18:00,20823.0,0.37,0.37,NaN,0.0,0.0,NaN,NaN
3,791,36,165660,241249.0,224329,2134-05-12 12:00:00,2134-05-12 12:19:00,20823.0,6.0,6.00,min,0.0,0.0,NaN,NaN
4,792,36,165660,241249.0,224330,2134-05-12 12:00:00,2134-05-12 12:19:00,20823.0,2.5,2.50,NaN,0.0,0.0,NaN,NaN


In [3]:
chartevents_cleaned = chartevents[chartevents['ICUSTAY_ID'].notna()]

Este código filtra las filas del DataFrame donde la columna ICUSTAY_ID tiene valores nulos (faltantes), asegurando que solo se mantengan los registros con identificadores válidos de estadía en UCI para el análisis.

In [ ]:
chartevents_cleaned['ICUSTAY_ID'] = chartevents_cleaned['ICUSTAY_ID'].astype(int)
icustays_cleaned['ICUSTAY_ID'] = icustays_cleaned['ICUSTAY_ID'].astype(int)

chartevents_cleaned= chartevents_cleaned[chartevents_cleaned['ICUSTAY_ID'].isin(icustays_cleaned['ICUSTAY_ID'])]

print(f"ICUSTAY_IDs únicos antes del filtro: {chartevents['ICUSTAY_ID'].nunique()}")
print(f"ICUSTAY_IDs únicos después del filtro: {chartevents_cleaned['ICUSTAY_ID'].nunique()}")

C:\Users\Ines\AppData\Local\Temp\ipykernel_868\2818262811.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chartevents_cleaned['ICUSTAY_ID'] = chartevents_cleaned['ICUSTAY_ID'].astype(int)


ICUSTAY_IDs únicos antes do filtro: 27241
ICUSTAY_IDs únicos após filtro: 27241


Este código tiene como objetivo comprender mejor el conjunto de datos chartevents garantizando coherencia con el conjunto de datos icustays. Convierte las columnas ICUSTAY_ID en ambos conjuntos de datos a enteros para evitar desajustes de tipo, luego filtra chartevents_cleaned para mantener solo los registros cuyo ICUSTAY_ID existe en icustays_cleaned. Finalmente, compara e imprime el número de identificadores únicos de estadía en UCI antes y después de este filtrado para mostrar cuántos eventos permanecen vinculados a estadías en UCI válidas.

In [5]:
unique_ids_in_icustays = set(icustays_cleaned['ICUSTAY_ID'].unique())
print(f"Total ICUSTAY_IDs in icustays_cleaned: {len(unique_ids_in_icustays)}")
print(len(icustays_cleaned))

Total ICUSTAY_IDs in icustays_cleaned: 61522
61522


Confirmando que cada línea tiene un 'ICUSTAY_ID' único.

In [6]:
chartevents_cleaned = chartevents_cleaned.drop(columns=['ROW_ID','STORETIME', 'CGID', 'VALUENUM','VALUEUOM','WARNING', 'ERROR', 'RESULTSTATUS', 'STOPPED'])

In [ ]:
# Filtrar diagnoses_icd_cleaned donde recode == '6'
hadm_ids_icd6 = diagnoses_icd_cleaned[diagnoses_icd_cleaned['recode'] == 6]['HADM_ID'].unique()

# Obtener los ICUSTAY_IDs correspondientes a estos HADM_IDs
icustays_ids_icd6 = icustays_cleaned[icustays_cleaned['HADM_ID'].isin(hadm_ids_icd6)]['ICUSTAY_ID'].unique()

# Filtrar chartevents basado en estos ICUSTAY_IDs
chartevents_icd6 = chartevents_cleaned[chartevents_cleaned['ICUSTAY_ID'].isin(icustays_ids_icd6)]

print(f"recode == '6' → HADM_IDs: {len(hadm_ids_icd6)}")
print(f"ICUSTAY_IDs correspondientes: {len(icustays_ids_icd6)}")
print(f"Registros filtrados en chartevents: {len(chartevents_icd6)}")

# Filtrar chartevents_icd6 para mantener solo estos ICUSTAY_IDs
chartevents_icd6 = chartevents_icd6[chartevents_icd6['ICUSTAY_ID'].isin(unique_ids_in_icustays)]
print(f"Registros filtrados en chartevents: {len(chartevents_icd6)}")


recode == '6' → HADM_IDs: 43243
Corresponding ICUSTAY_IDs: 45795
Filtered records in chartevents: 34586640
Filtered records in chartevents: 34586640


Este código filtra los datos clínicos para enfocarse en pacientes con una categoría de diagnóstico específica (recode == 6). Primero, extrae los identificadores de admisión hospitalaria (HADM_ID) para diagnósticos con recode 6. Luego, encuentra los identificadores de estadía en UCI (ICUSTAY_ID) correspondientes vinculados a esas admisiones. Utilizando estos identificadores de estadía en UCI, filtra el conjunto de datos chartevents_cleaned para mantener solo los registros relacionados con estos pacientes. Finalmente, filtra aún más para incluir solo las estadías en UCI presentes en una lista predefinida (unique_ids_in_icustays). Los conteos impresos muestran el número de admisiones relevantes, estadías en UCI y registros de eventos clínicos filtrados en cada paso.

In [ ]:
# Filtrar filas donde el VALUE es una cadena
chartevents_str_itemid = chartevents_icd6[chartevents_icd6['VALUE'].apply(lambda x: isinstance(x, str))]

# Ver los valores únicos de estas cadenas
print(chartevents_str_itemid['VALUE'].unique())

chartevents_icd6['VALUE'] = pd.to_numeric(chartevents_icd6['VALUE'], errors='coerce').astype('float64')

# Filtrar filas donde el VALUE es una cadena
chartevents_str_itemid = chartevents_icd6[chartevents_icd6['VALUE'].apply(lambda x: isinstance(x, str))]

# Ver los valores únicos de estas cadenas
print(chartevents_str_itemid['VALUE'].unique())


['91.4' '0' '170' ... '506.12200927734375' '921.9329833984375'
 '12.06089973449707']
[]


Este código primero filtra chartevents_icd6 para encontrar filas donde la columna VALUE contiene datos de cadena, mostrando los valores de cadena únicos encontrados. Luego, intenta convertir todas las entradas VALUE a valores numéricos, coercionando cualquier entrada no convertible a NaN. Después de esta conversión, verifica nuevamente si hay valores de cadena restantes en VALUE e imprime sus ocurrencias únicas, ayudando a identificar y manejar datos no numéricos dentro de esta columna.

In [ ]:
# Verificar si hay ceros en VALUE
print((chartevents_icd6['VALUE'] == 0).any())

# Calcular el valor medio para cada ITEMID
mean_values = chartevents_icd6.groupby('ITEMID')['VALUE'].mean()

# Función para rellenar valores faltantes
def fill_value(row):
    if pd.isna(row['VALUE']):
        return mean_values.get(row['ITEMID'], np.nan)  
    else:
        return row['VALUE']
def fill_value(row):
    if pd.isna(row['VALUE']):
        return mean_values.get(row['ITEMID'], np.nan) 
    else:
        return row['VALUE']

print(chartevents_icd6['VALUE'].isna().sum())

# Rellenar valores faltantes con la media por ITEMIDchartevents_icd6['VALUE'] = chartevents_icd6['VALUE'].fillna(chartevents_icd6['ITEMID'].map(mean_values))    

True
2000537


Este código primero verifica si hay valores cero en la columna VALUE de chartevents_icd6. Luego, calcula el valor medio para cada ITEMID. A continuación, define una función fill_value para reemplazar valores faltantes (NaN) en VALUE con la media correspondiente de ITEMID. Sin embargo, en lugar de usar esta función explícitamente, utiliza fillna de pandas combinado con map para rellenar directamente los valores NaN con la media para cada ITEMID. Finalmente, imprime el conteo de valores faltantes restantes en VALUE después de esta imputación, mostrando cuántos NaNs permanecen.

In [ ]:
# Mostrar el número de ICUSTAY_IDs e ITEMIDs únicos
print(f"ICUSTAY_IDs únicos: {chartevents_icd6['ICUSTAY_ID'].nunique()}")
print(f"ITEMIDs únicos: {chartevents_icd6['ITEMID'].nunique()}")

ICUSTAY_IDs únicos: 23535
ITEMIDs únicos: 1578


In [11]:
print(chartevents_icd6.dtypes)


SUBJECT_ID      int64
HADM_ID         int64
ICUSTAY_ID      int64
ITEMID          int64
CHARTTIME      object
VALUE         float64
dtype: object


In [ ]:
# Agrupar por ICUSTAY_ID e ITEMID y calcular el valor promedio
df_grouped = chartevents_icd6.groupby(['ICUSTAY_ID', 'ITEMID'])['VALUE'].mean()

print(df_grouped.head(20))

# Pivotar los datos para crear una tabla con ITEMIDs como columnas
df_pivot = df_grouped.unstack()


print(df_pivot.head())print(len(df_pivot))

ICUSTAY_ID  ITEMID
200001      220045     84.616162
            220046    120.000000
            220047     58.571429
            220050    109.600000
            220051     54.184615
            220052     83.686567
            220056     82.500000
            220058    147.500000
            220179    102.129032
            220180     56.064516
            220181     66.870968
            220210     20.535354
            220224     95.000000
            220227     92.000000
            220228      7.766667
            220235     42.428571
            220277     98.354167
            220545     24.600000
            220546      3.233333
            220587     29.000000
Name: VALUE, dtype: float64
ITEMID      1       2       3       24      25      26      27      28      \
ICUSTAY_ID                                                                   
200001         NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
200010         NaN     NaN     NaN     NaN     NaN     NaN 

Este código agrupa los datos de chartevents_icd6 por ICUSTAY_ID e ITEMID, calculando el valor promedio para cada combinación. Luego imprime los primeros 20 resultados agrupados. Luego, reorganiza los datos agrupados en una tabla pivote donde cada fila representa una estadía en UCI (ICUSTAY_ID), cada columna corresponde a un ITEMID, y las celdas contienen los valores promedio para esos elementos durante esa estadía. Finalmente, imprime las primeras filas de esta tabla pivote y el número total de estadías en UCI (filas) en el conjunto de datos pivotado.

In [ ]:
# Contar valores únicos en cada columna
nunique_per_column = df_pivot.nunique(dropna=True)

# Mantener solo columnas con más de un valor único
cols_to_keep = nunique_per_column[nunique_per_column > 1].index

df_pivot_filtered = df_pivot[cols_to_keep]


print(f"Columnas antes: {df_pivot.shape[1]}")print(f"Columnas después de eliminar constantes: {df_pivot_filtered.shape[1]}")

Columns before: 1578
Columns after removing constant ones: 852


Este código cuenta el número de valores únicos (no NaN) en cada columna del DataFrame pivotado df_pivot. Luego selecciona solo las columnas que tienen más de un valor único, eliminando efectivamente las columnas que son constantes (el mismo valor para todas las estadías en UCI). El DataFrame filtrado df_pivot_filtered mantiene solo estas columnas variables. Finalmente, imprime el número de columnas antes y después de este filtrado para mostrar cuántas columnas constantes fueron eliminadas.

In [ ]:
# Rellenar valores faltantes con la media de la columna
df_pivot_filtered = df_pivot_filtered.fillna(df_pivot_filtered.mean())
df_pivot_filtered.head()

ITEMID,2,3,25,26,29,50,52,55,59,60,...,228375,228376,228377,228378,228379,228380,228381,228382,228392,228444
ICUSTAY_ID,,,,,,,,,,,,,,,,,,,,,
200001,0.985331,0.955861,164.428793,259.747692,-35.527874,20.411955,80.727524,94.019236,109.490318,3.618011,...,35.437574,17.198265,10.726075,-0.431593,1.086753,65.633333,1045.946411,1919.428812,25.911111,40.935808
200010,0.985331,0.955861,164.428793,259.747692,-35.527874,20.411955,80.727524,94.019236,109.490318,3.618011,...,35.437574,17.198265,10.726075,-0.431593,1.086753,65.633333,1045.946411,1919.428812,25.911111,40.935808
200011,0.985331,0.955861,164.428793,259.747692,-35.527874,20.411955,80.727524,94.019236,109.490318,3.618011,...,35.437574,17.198265,10.726075,-0.431593,1.086753,65.633333,1045.946411,1919.428812,25.911111,40.935808
200016,0.985331,0.955861,164.428793,259.747692,-35.527874,20.411955,80.727524,94.019236,109.490318,3.618011,...,35.437574,17.198265,10.726075,-0.431593,1.086753,65.633333,1045.946411,1919.428812,25.911111,40.935808
200021,0.985331,0.955861,164.428793,259.747692,-35.527874,20.411955,80.727524,94.019236,109.490318,3.618011,...,35.437574,17.198265,10.726075,-0.431593,1.086753,65.633333,1045.946411,1919.428812,25.911111,40.935808


In [ ]:
# Fusionar con la duración de la estadía (LOS)
df = df_pivot_filtered.merge(icustays_cleaned[['ICUSTAY_ID', 'LOS']], on='ICUSTAY_ID', how='inner')
print(len(df))

23535


Considerando 24 horas:

In [ ]:
# Convertir a formato datetime
chartevents_icd6['CHARTTIME'] = pd.to_datetime(chartevents_icd6['CHARTTIME'], format='mixed', errors='coerce')
icustays_cleaned['INTIME'] = pd.to_datetime(icustays_cleaned['INTIME'])

# Redondear al minuto más cercano
chartevents_icd6['CHARTTIME']= chartevents_icd6['CHARTTIME'].dt.floor('min')
icustays_cleaned['INTIME']= icustays_cleaned['INTIME'].dt.floor('min')

# Fusionar para obtener el tiempo de inicio
df_merged = chartevents_icd6.merge(
    icustays_cleaned[['ICUSTAY_ID', 'INTIME']],
    on='ICUSTAY_ID',
    how='left'
)

# Filtrar para observaciones dentro de las primeras 24 horas
df_merged = df_merged[df_merged['CHARTTIME'] <= df_merged['INTIME'] + pd.Timedelta(hours=24)]
print(df_merged.columns)

# Filtrar df para mantener solo los ICUSTAY_ID que están en df_merged

df = df[df['ICUSTAY_ID'].isin(df_merged['ICUSTAY_ID'].unique())]
print(len(df))

Index(['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ITEMID', 'CHARTTIME', 'VALUE',
       'INTIME'],
      dtype='object')
23492


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separar características (X) y variable objetivo (y)
X = df.drop(columns=['LOS', 'ICUSTAY_ID']) 
y = df['LOS']

# División entrenamiento/prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Crear y entrenar modelo
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Realizar predicciones
y_pred = model.predict(X_test_scaled)

# Evaluar el modelo
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE: {mae:.2f}")   
print(f"RMSE: {rmse:.2f}")


MAE: 1.56
RMSE: 3.24


Considerando 12 horas:

In [ ]:
# Filtrar para observaciones dentro de las primeras 12 horas
df_merged = df_merged[df_merged['CHARTTIME'] <= df_merged['INTIME'] + pd.Timedelta(hours=12)]
print(df_merged.columns)

# Filtrar df para mantener solo los ICUSTAY_ID que están en df_merged

df = df[df['ICUSTAY_ID'].isin(df_merged['ICUSTAY_ID'].unique())]print(len(df))

Index(['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ITEMID', 'CHARTTIME', 'VALUE',
       'INTIME'],
      dtype='object')
23413


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['LOS', 'ICUSTAY_ID']) 
y = df['LOS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [21]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE: {mae:.2f}")   
print(f"RMSE: {rmse:.2f}")


MAE: 1.56
RMSE: 3.17


Considerando 72 horas:

In [ ]:
# Filtrar para observaciones dentro de las primeras 72 horas
df_merged = df_merged[df_merged['CHARTTIME'] <= df_merged['INTIME'] + pd.Timedelta(hours=72)]
print(df_merged.columns)

# Filtrar df para mantener solo los ICUSTAY_ID que están en df_merged

df = df[df['ICUSTAY_ID'].isin(df_merged['ICUSTAY_ID'].unique())]print(len(df))

Index(['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ITEMID', 'CHARTTIME', 'VALUE',
       'INTIME'],
      dtype='object')
23413


In [23]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['LOS', 'ICUSTAY_ID']) 
y = df['LOS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [24]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE: {mae:.2f}")   
print(f"RMSE: {rmse:.2f}")


MAE: 1.56
RMSE: 3.17


Los resultados fueron bastante similares en las diversas ventanas de tiempo que probamos; sin embargo, encontramos que la ventana de 24 horas produjo los peores resultados.